# Часть DL-1. Классификация фотографий квартир по уровню ремонта

Мы хотим по фотографиям, приложенным к объявлению с продажей квартир, определить качество или уровень ремонта.
(признак `rating` с пятью классами). Метка стоит на объявлении, обучаемся по отдельным фото, потом усредняем предсказания по объявлению.

Подключаем пак фотографий с Google Drive, полученный в ноутбуке photo_pack

А также далее работаем с таблицей с рейтингами, собранную с помощью парсинга и мучительно размеченную вручную

In [1]:
!pip install -q timm torchinfo

from google.colab import drive
drive.mount('/content/drive')

cian_photo = '/content/drive/MyDrive/cian/cian_photos_224.npz'

Mounted at /content/drive


Импорты, функция `seed_everything` и конфиг эксперимента

In [33]:
import os
import io
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchinfo import summary
import timm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class CFG:
    n_classes = 5
    num_epochs = 3
    batch_size = 32
    lr = 0.001
    seed = 67


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

seed_everything(CFG.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


Загружаем пак фотографий и таблицу, делим выборку по айдишникам объявлений, чтобы фото одной квартиры не попали и в тест, и в трейн.

In [4]:
df = pd.read_csv('df_merged_ratings_new.csv')

pack = np.load(cian_photo)
blob = pack['blob']
offsets = pack['offsets']
pack_lids = pack['listing_ids']

id2rating = dict(zip(df['id'].astype(int), df['rating'].astype(int)))

photos = []
for i, lid in enumerate(pack_lids):
    lid = int(lid)
    if lid in id2rating:
        photos.append((i, lid, id2rating[lid] - 1))

groups = np.array([p[1] for p in photos])
indices = np.arange(len(photos))

splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=CFG.seed)
train_val_idx, test_idx = next(splitter.split(indices, groups=groups))

splitter_val = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=CFG.seed)
train_rel, val_rel = next(splitter_val.split(train_val_idx, groups=groups[train_val_idx]))
train_idx = train_val_idx[train_rel]
val_idx = train_val_idx[val_rel]


def make_items(idx_list):
    return [(photos[i][0], photos[i][2]) for i in idx_list]


train_items = make_items(train_idx)
val_items = make_items(val_idx)
test_items = make_items(test_idx)

print('train:', len(train_items), 'val:', len(val_items), 'test:', len(test_items))

train: 28634 val: 6040 test: 6235


### Аугментации для train

In [5]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class PhotoDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        pack_idx, label = self.items[index]
        raw = blob[offsets[pack_idx]:offsets[pack_idx + 1]].tobytes()
        image = Image.open(io.BytesIO(raw)).convert('RGB')
        return self.transform(image), label


train_loader = DataLoader(PhotoDataset(train_items, train_transform),
                          batch_size=CFG.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(PhotoDataset(val_items, val_transform),
                        batch_size=CFG.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(PhotoDataset(test_items, val_transform),
                         batch_size=CFG.batch_size, shuffle=False, num_workers=2)

counts = Counter(label for _, label in train_items)
weights = [len(train_items) / (CFG.n_classes * counts.get(c, 1)) for c in range(CFG.n_classes)]
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights).float().to(DEVICE))

Для логирования обучения глубинных сетей будем использовать библиотеку MLflow. Гайд по использованию брали здесь:
1. https://mlflow.org/docs/latest/ml/tracking/quickstart/
2. https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pytorch.html

In [6]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.9/936.9 kB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
!git clone https://github.com/iaportnov/GP5_VERY_DEEP_LEARNING.git

Cloning into 'GP5_VERY_DEEP_LEARNING'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 85 (delta 29), reused 26 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 38.31 MiB | 6.93 MiB/s, done.
Resolving deltas: 100% (29/29), done.
Updating files: 100% (24/24), done.


In [23]:
%cd GP5_VERY_DEEP_LEARNING

[Errno 2] No such file or directory: 'GP5_VERY_DEEP_LEARNING'
/content/GP5_VERY_DEEP_LEARNING


In [13]:
import copy

In [24]:
import mlflow
import mlflow.pytorch
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.create_experiment(name='ЦЫГАН ЭКСПЕРИМЕНТ 5', artifact_location='file:./logs')
mlflow.set_experiment('ЦЫГАН ЭКСПЕРИМЕНТ 5')

<Experiment: artifact_location='file:///content/GP5_VERY_DEEP_LEARNING/logs', creation_time=1781601031908, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1781601031908, lifecycle_stage='active', name='ЦЫГАН ЭКСПЕРИМЕНТ 5', tags={}, trace_location=None, workspace='default'>

Функции обучения и инференса, и цикл по эпохам.

In [19]:
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train()
    train_loss = 0
    correct = 0
    for data, target in tqdm(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        train_loss = criterion(output, target)
        train_loss.backward()
        optimizer.step()
    train_accuracy = correct / len(test_loader.dataset)
    tqdm.write('\nTrain set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        train_loss, 100. * correct / len(train_loader.dataset)))
    return train_loss.item(), train_accuracy


def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_accuracy = correct / len(test_loader.dataset)
    tqdm.write('Test set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        test_loss, 100. * correct / len(test_loader.dataset)))
    return test_loss.item(), test_accuracy

def main(model):
    seed_everything(CFG.seed)
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=CFG.lr)

    best_accuracy = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    with mlflow.start_run():
      mlflow.log_params({
          'Задача': 'Работа с фото',
          'model': model.__class__.__name__,
          'criterion': criterion.__class__.__name__,
          'optimizer': optimizer.__class__.__name__,
          'epochs': CFG.num_epochs
      })
      for epoch in range(1, CFG.num_epochs + 1):
          print('\nEpoch:', epoch)
          train_loss, train_accuracy = train(model, DEVICE, train_loader, optimizer, criterion, epoch)
          test_loss, test_accuracy = test(model, DEVICE, val_loader, criterion)

          mlflow.log_metric('loss/train', train_loss, step=epoch)
          mlflow.log_metric('loss/val', test_loss, step=epoch)
          mlflow.log_metric('accuracy/train', train_accuracy, step=epoch)
          mlflow.log_metric('accuracy/val', test_accuracy, step=epoch)

          if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            best_weights = copy.deepcopy(model.state_dict())

      model.load_state_dict(best_weights)

      torch.save(best_weights, f'{model.__class__.__name__}.pt')

      mlflow.log_metric('best_val_accuracy', best_accuracy)
      mlflow.pytorch.log_model(model, 'model')

      print('Training is ended!')
    return model

### Модель 1 - CNN

In [16]:
class Renovation_CNN(nn.Module):
    def __init__(self, n_classes=CFG.n_classes):
        super(Renovation_CNN, self).__init__()

        self.batch_norm0 = nn.BatchNorm2d(3)

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.batch_norm1 = nn.BatchNorm2d(32)
        self.act1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.batch_norm2 = nn.BatchNorm2d(64)
        self.act2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.batch_norm3 = nn.BatchNorm2d(128)
        self.act3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)

        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)
        self.batch_norm4 = nn.BatchNorm2d(128)
        self.act4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 14 * 14, 256)
        self.batch_norm5 = nn.BatchNorm1d(256)
        self.act5 = nn.Tanh()
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 64)
        self.act6 = nn.Tanh()
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.batch_norm0(x)

        x = self.conv1(x)
        x = self.batch_norm1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.batch_norm2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.batch_norm3(x)
        x = self.act3(x)
        x = self.pool3(x)

        x = self.conv4(x)
        x = self.batch_norm4(x)
        x = self.act4(x)
        x = self.pool4(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.batch_norm5(x)
        x = self.act5(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.act6(x)
        x = self.dropout2(x)
        x = self.fc3(x)

        return x


model_cnn = Renovation_CNN().to(DEVICE)
summary(model=model_cnn,
        input_size=(CFG.batch_size, 3, 224, 224),
        col_names=['input_size', 'output_size', 'num_params', 'trainable'],
        col_width=20)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
Renovation_CNN                           [32, 3, 224, 224]    [32, 5]              --                   True
├─BatchNorm2d: 1-1                       [32, 3, 224, 224]    [32, 3, 224, 224]    6                    True
├─Conv2d: 1-2                            [32, 3, 224, 224]    [32, 32, 224, 224]   896                  True
├─BatchNorm2d: 1-3                       [32, 32, 224, 224]   [32, 32, 224, 224]   64                   True
├─ReLU: 1-4                              [32, 32, 224, 224]   [32, 32, 224, 224]   --                   --
├─MaxPool2d: 1-5                         [32, 32, 224, 224]   [32, 32, 112, 112]   --                   --
├─Conv2d: 1-6                            [32, 32, 112, 112]   [32, 64, 112, 112]   18,496               True
├─BatchNorm2d: 1-7                       [32, 64, 112, 112]   [32, 64, 112, 112]   128                  True
├─ReLU: 1-8       

In [34]:
model_cnn = main(model_cnn)


Epoch: 1


  0%|          | 0/895 [00:00<?, ?it/s]


Train set: Average loss: 1.5437, Accuracy: 41.31%
Test set: Average loss: 0.8150, Accuracy: 43.82%

Epoch: 2


  0%|          | 0/895 [00:00<?, ?it/s]


Train set: Average loss: 1.0079, Accuracy: 39.57%
Test set: Average loss: 0.9555, Accuracy: 36.59%

Epoch: 3


...


Train set: Average loss: 1.2111, Accuracy: 41.02%


2026/06/16 09:22:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:22:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Test set: Average loss: 1.2228, Accuracy: 41.66%


2026/06/16 09:22:55 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 09:23:31 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Training is ended!


### Модель 2 - ResNet18

In [35]:
resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for param in resnet.parameters():
    param.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, CFG.n_classes)

summary(model=resnet,
        input_size=(CFG.batch_size, 3, 224, 224),
        col_names=['input_size', 'output_size', 'num_params', 'trainable'],
        col_width=20)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 224, 224]    [32, 5]              --                   Partial
├─Conv2d: 1-1                            [32, 3, 224, 224]    [32, 64, 112, 112]   (9,408)              False
├─BatchNorm2d: 1-2                       [32, 64, 112, 112]   [32, 64, 112, 112]   (128)                False
├─ReLU: 1-3                              [32, 64, 112, 112]   [32, 64, 112, 112]   --                   --
├─MaxPool2d: 1-4                         [32, 64, 112, 112]   [32, 64, 56, 56]     --                   --
├─Sequential: 1-5                        [32, 64, 56, 56]     [32, 64, 56, 56]     --                   False
│    └─BasicBlock: 2-1                   [32, 64, 56, 56]     [32, 64, 56, 56]     --                   False
│    │    └─Conv2d: 3-1                  [32, 64, 56, 56]     [32, 64, 56, 56]     (36,864)             False
│    │    

In [36]:
resnet = main(resnet)


Epoch: 1


  0%|          | 0/895 [00:00<?, ?it/s]


Train set: Average loss: 1.0833, Accuracy: 42.98%
Test set: Average loss: 0.5703, Accuracy: 42.76%

Epoch: 2


  0%|          | 0/895 [00:00<?, ?it/s]


Train set: Average loss: 0.9337, Accuracy: 46.93%
Test set: Average loss: 0.5139, Accuracy: 51.26%

Epoch: 3


  0%|          | 0/895 [00:00<?, ?it/s]


Train set: Average loss: 0.9635, Accuracy: 48.20%
Test set: Average loss: 0.3786, Accuracy: 42.35%


2026/06/16 09:31:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:31:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 09:31:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 09:31:51 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

Training is ended!


### Модель 3 - EfficientNet-B0
`timm` - библиотека, которую нашли как хороший выбор для работы с компьютерным зрением.

Замораживаем основной слой, так как у нас недостаточно данных для полноценного обучения и без этого быстро случится переобучение

In [37]:
effnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=CFG.n_classes)
for param in effnet.parameters():
    param.requires_grad = False
for param in effnet.get_classifier().parameters():
    param.requires_grad = True

summary(model=effnet,
        input_size=(CFG.batch_size, 3, 224, 224),
        col_names=['input_size', 'output_size', 'num_params', 'trainable'],
        col_width=20)

Layer (type:depth-idx)                        Input Shape          Output Shape         Param #              Trainable
EfficientNet                                  [32, 3, 224, 224]    [32, 5]              --                   Partial
├─Conv2d: 1-1                                 [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
├─BatchNormAct2d: 1-2                         [32, 32, 112, 112]   [32, 32, 112, 112]   64                   False
│    └─Identity: 2-1                          [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─SiLU: 2-2                              [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
├─Sequential: 1-3                             [32, 32, 112, 112]   [32, 320, 7, 7]      --                   False
│    └─Sequential: 2-3                        [32, 32, 112, 112]   [32, 16, 112, 112]   --                   False
│    │    └─DepthwiseSeparableConv: 3-1       [32, 32, 112, 112]   [32, 16, 112,

In [38]:
effnet = main(effnet)


Epoch: 1


  0%|          | 0/895 [00:00<?, ?it/s]


Train set: Average loss: 3.1917, Accuracy: 34.95%
Test set: Average loss: 1.2750, Accuracy: 38.63%

Epoch: 2


  0%|          | 0/895 [00:00<?, ?it/s]


Train set: Average loss: 0.9468, Accuracy: 42.09%
Test set: Average loss: 1.0112, Accuracy: 41.13%

Epoch: 3


...


Train set: Average loss: 1.2120, Accuracy: 44.59%


2026/06/16 09:40:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Test set: Average loss: 0.8644, Accuracy: 40.68%


2026/06/16 09:40:56 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 09:40:56 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Training is ended!


### Сравнение моделей на тесте
Считаем accuracy и macro-F1 на отложенной выборке, мы макрофаги макрофаги

In [39]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    y_true = []
    y_pred = []
    with torch.no_grad():
        for data, target in loader:
            output = model(data.to(DEVICE))
            pred = output.argmax(dim=1).cpu()
            correct += pred.eq(target).sum().item()
            y_true.extend(target.tolist())
            y_pred.extend(pred.tolist())
    accuracy = 100. * correct / len(loader.dataset)
    return accuracy, y_true, y_pred


models = {'CNN': model_cnn, 'ResNet18': resnet, 'EfficientNet': effnet}
results = {}
for name, model in models.items():
    accuracy, y_true, y_pred = evaluate(model, test_loader)
    results[name] = {'accuracy_%': round(accuracy, 2),
                     'macro_f1': round(f1_score(y_true, y_pred, average='macro'), 3)}

comparison = pd.DataFrame(results).T.sort_values('macro_f1', ascending=False)
comparison

,accuracy_%,macro_f1
ResNet18,53.15,0.560
CNN,46.27,0.478
EfficientNet,43.19,0.471


### ЧТО ЖЕ МЫ НАДЕЛАЛИ И ПОЧЕМУ?
1. Используем Оптимизатор Adam для подбора и адаптации шага для каждого параметра модели
2. Берем Взвешенную CrossEntropy, потому что классы несбалансированы (у нас при оценке получилось мало требуется ремонт и современный), без нее модель сильно смещается к большому классу
3. Аугментация также из-за нехватки данных, так мы боремся с переобучением
4. batch_size = 32 взяли из семинара, но вообще мы не тратим с таким размером много ресурсов GPU
5. num_epochs = 3 во избежании переобучения, прогоняли и на 10, но на последних эпохах качество конкретно страдало
6. Замораживаем backbone у ResNet18 и EfficientNet-B0, так как на нескольких тысячах объявлений дообучение всей сети переобучается, поэтому обучаем только классификатор

**ЕЩКЕРЕ**

## Грузим на гитхаб

In [40]:
!git config --global user.name 'iaportnov'
!git config --global user.email 'iaportnov2004@gmail.com'

In [41]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	EfficientNet.pt
	Renovation_CNN.pt
	ResNet.pt
	logs/models/m-863b7bafe2024647b7240195fc4e00c6/
	logs/models/m-d848ea6b14884bb294a87ca031f9af13/
	logs/models/m-e458e47908ab48d285d59f22fd1be40c/

nothing added to commit but untracked files present (use "git add" to track)


In [42]:
!git add mlflow.db logs/

In [43]:
!git commit -m "Some new DL-models"

[main 7d5e66a] Some new DL-models
 18 files changed, 212 insertions(+)
 create mode 100644 logs/models/m-863b7bafe2024647b7240195fc4e00c6/artifacts/MLmodel
 create mode 100644 logs/models/m-863b7bafe2024647b7240195fc4e00c6/artifacts/conda.yaml
 create mode 100644 logs/models/m-863b7bafe2024647b7240195fc4e00c6/artifacts/data/model.pth
 create mode 100644 logs/models/m-863b7bafe2024647b7240195fc4e00c6/artifacts/data/pickle_module_info.txt
 create mode 100644 logs/models/m-863b7bafe2024647b7240195fc4e00c6/artifacts/python_env.yaml
 create mode 100644 logs/models/m-863b7bafe2024647b7240195fc4e00c6/artifacts/requirements.txt
 create mode 100644 logs/models/m-d848ea6b14884bb294a87ca031f9af13/artifacts/MLmodel
 create mode 100644 logs/models/m-d848ea6b14884bb294a87ca031f9af13/artifacts/conda.yaml
 create mode 100644 logs/models/m-d848ea6b14884bb294a87ca031f9af13/artifacts/data/model.pth
 create mode 100644 logs/models/m-d848ea6b14884bb294a87ca031f9af13/artifacts/data/pickle_module_info.txt
 c

In [44]:
import os
from getpass import getpass

token = getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

GitHub token: ··········


In [45]:
!git remote set-url origin https://$token@github.com/iaportnov/GP5_VERY_DEEP_LEARNING.git
!git push origin main

To https://github.com/iaportnov/GP5_VERY_DEEP_LEARNING.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/iaportnov/GP5_VERY_DEEP_LEARNING.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [46]:
!git pull --rebase origin main
!git push origin main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 7 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 3.79 MiB | 4.86 MiB/s, done.
From https://github.com/iaportnov/GP5_VERY_DEEP_LEARNING
 * branch            main       -> FETCH_HEAD
   3f2ba20..62bfe2e  main       -> origin/main
Successfully rebased and updated refs/heads/main.
Enumerating objects: 30, done.
Counting objects: 100% (30/30), done.
Delta compression using up to 2 threads
Compressing objects: 100% (22/22), done.
Writing objects: 100% (27/27), 77.74 MiB | 13.44 MiB/s, done.
Total 27 (delta 6), reused 2 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (6/6), completed with 1 local object.
To https://github.com/iaportnov/GP5_VERY_DEEP_LEARNING.git
   62bfe2e..da21682  main -> main


## Выводы по первой части DL

Перенос обучения ResNet18 ожидаемо превосходит CNN по accuracy и F1, так как признаки ImageNet работают на небольшом датасете, а сеть с нуля переобучается

Среди transfer-моделей при сопоставимом обучении ResNet18 оказался точнее EfficientNet-B0, что в целом объяснимо бОльшим числом параметров, поэтому именно эту модель можно назвать лучшей для оценки ремонта, пусть качество и не идельное

CNN оставляем как бейзлайн для сравнения с трансфер-лернингом